In [7]:
import re
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

random.seed(42)
np.random.seed(42)
sns.set_style("whitegrid")

SENTIMENT_COLORS = {"Positive": "#2ecc71", "Neutral": "#95a5a6", "Negative": "#e74c3c"}

In [17]:
#social media comments
PLATFORMS = ["Twitter", "Instagram", "Facebook", "YouTube"]
PRODUCTS = ["SmartWatch X2", "EcoBottle", "AirPods Clone Pro", "FitBand 3", "UltraBlender"]

POSITIVE_TEMPLATES = [
    "I absolutely love my new {product}! Best purchase this year #amazing",
    "{product} exceeded my expectations, battery life is incredible",
    "Just got the {product} and I'm blown away by the quality!",
    "Customer service was fantastic when I asked about {product} #greatservice",
    "The {product} design is so sleek, totally worth the price",
    "Been using {product} for a week, works perfectly every time",
    "Highly recommend the {product} to anyone looking for quality",
    "{product} is a game changer, super happy with this purchase",
    "Fast shipping and the {product} works great, five stars!",
    "This {product} campaign convinced me to buy and I don't regret it",
]
NEGATIVE_TEMPLATES = [
    "My {product} stopped working after two days, very disappointed",
    "Terrible experience with {product}, customer support was no help",
    "{product} is way overpriced for what you actually get",
    "Not happy with the {product}, battery drains too fast #frustrated",
    "The {product} broke within a week, waste of money",
    "Worst purchase ever, {product} feels cheap and flimsy",
    "Still waiting for a refund on my {product}, awful service",
    "{product} does not match the description at all, misleading ad",
    "Regret buying the {product}, quality control is bad",
    "Delivery took forever and the {product} arrived damaged",
]
NEUTRAL_TEMPLATES = [
    "Anyone know when {product} restocks in blue?",
    "Does {product} come with a warranty?",
    "Comparing {product} with other brands before I decide",
    "Saw the {product} ad on TV, looks interesting",
    "What's the battery life on the {product} exactly?",
    "Is {product} available internationally?",
    "Thinking about getting {product} for a gift",
    "{product} launch event was today, watched the livestream",
    "Can someone share their honest review of {product}?",
    "Waiting for {product} price to drop during the sale",
]


def collect_comments(n=400):
    rows = []
    start_date = pd.Timestamp("2026-05-01")
    for i in range(n):
        product = random.choice(PRODUCTS)
        bucket = random.choices(["pos", "neg", "neu"], weights=[0.45, 0.30, 0.25])[0]
        template = random.choice(
            POSITIVE_TEMPLATES if bucket == "pos" else NEGATIVE_TEMPLATES if bucket == "neg" else NEUTRAL_TEMPLATES
        )
        comment = template.format(product=product)
        if random.random() < 0.3:
            comment += random.choice(["Happy", "Anger", "Fire", "Loved", "confused"])
        if random.random() < 0.15:
            comment += " @CompanySupport"
        if random.random() < 0.1:
            comment += " http://bit.ly/promo123"

        rows.append({
            "comment_id": i + 1,
            "username": f"user_{random.randint(1000, 9999)}",
            "platform": random.choice(PLATFORMS),
            "timestamp": start_date + pd.Timedelta(days=random.randint(0, 60), hours=random.randint(0, 23)),
            "product": product,
            "comment": comment,
            "likes": np.random.poisson(15),
        })
    return pd.DataFrame(rows).sort_values("timestamp").reset_index(drop=True)

In [18]:
#Clean and preprocessed text
STOPWORDS = set("""
a about above after again against all am an and any are aren't as at be
because been before being below between both but by can't cannot could
couldn't did didn't do does doesn't doing don't down during each few for
from further had hadn't has hasn't have haven't having he he'd he'll he's
her here here's hers herself him himself his how how's i i'd i'll i'm i've
if in into is isn't it it's its itself let's me more most mustn't my myself
no nor not of off on once only or other ought our ours ourselves out over
own same shan't she she'd she'll she's should shouldn't so some such than
that that's the their theirs them themselves then there there's these they
they'd they'll they're they've this those through to too under until up
very was wasn't we we'd we'll we're we've were weren't what what's when
when's where where's which while who who's whom why why's with won't would
wouldn't you you'd you'll you're you've your yours yourself yourselves
""".split())


def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^\x00-\x7F]+", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = [w for w in text.split() if w not in STOPWORDS and len(w) > 2]
    return " ".join(tokens)


def preprocess(df):
    df = df.copy()
    df["clean_comment"] = df["comment"].apply(clean_text)
    return df[df["clean_comment"].str.len() > 0].reset_index(drop=True)


In [19]:
#analysis
analyzer = SentimentIntensityAnalyzer()


def score_sentiment(text):
    compound = analyzer.polarity_scores(text)["compound"]
    if compound >= 0.05:
        label = "Positive"
    elif compound <= -0.05:
        label = "Negative"
    else:
        label = "Neutral"
    return label, compound


def add_sentiment(df):
    df = df.copy()
    results = df["comment"].apply(score_sentiment)
    df["sentiment"] = results.apply(lambda x: x[0])
    df["sentiment_score"] = results.apply(lambda x: x[1])
    return df

In [20]:
#trending topics
def find_top_keywords(df, n=15):
    vectorizer = TfidfVectorizer(max_features=500)
    tfidf_matrix = vectorizer.fit_transform(df["clean_comment"])
    scores = tfidf_matrix.sum(axis=0).A1
    terms = vectorizer.get_feature_names_out()
    keyword_df = pd.DataFrame({"keyword": terms, "score": scores}) \
        .sort_values("score", ascending=False).head(n).reset_index(drop=True)
    return keyword_df, vectorizer, tfidf_matrix


def find_topics(tfidf_matrix, vectorizer, n_topics=4, n_words=8):
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42, max_iter=20)
    lda.fit(tfidf_matrix)
    terms = vectorizer.get_feature_names_out()
    topics = {}
    for idx, topic in enumerate(lda.components_):
        topics[f"Topic {idx + 1}"] = [terms[i] for i in topic.argsort()[-n_words:][::-1]]
    return topics


In [29]:
#Visualize public opinion

sns.set_style("whitegrid")
SENTIMENT_COLORS = {"Positive": "#2ecc71", "Neutral": "#95a5a6", "Negative": "#e74c3c"}


def build_dashboard(df, keyword_df, out_path="opinion_dashboard.png"):
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    order = ["Positive", "Neutral", "Negative"]

    fig, axes = plt.subplots(3, 2, figsize=(16, 16))

    sentiment_counts = df["sentiment"].value_counts().reindex(order)
    axes[0, 0].bar(sentiment_counts.index, sentiment_counts.values,
                    color=[SENTIMENT_COLORS[s] for s in sentiment_counts.index])
    axes[0, 0].set_title("Overall Sentiment Distribution")
    axes[0, 0].set_ylabel("Number of Comments")

    product_sentiment = pd.crosstab(df["product"], df["sentiment"], normalize="index")[order]
    product_sentiment.plot(kind="bar", stacked=True, ax=axes[0, 1],
                            color=[SENTIMENT_COLORS[s] for s in order])
    axes[0, 1].set_title("Sentiment Share by Product")
    axes[0, 1].set_ylabel("Proportion")
    axes[0, 1].tick_params(axis="x", rotation=30)
    axes[0, 1].legend(title="Sentiment")

    top10 = keyword_df.head(10).sort_values("score")
    axes[1, 0].barh(top10["keyword"], top10["score"], color="#3498db")
    axes[1, 0].set_title("Top Trending Keywords (TF-IDF)")
    axes[1, 0].set_xlabel("TF-IDF Score")

    text_blob = " ".join(df["clean_comment"].astype(str))
    wc = WordCloud(width=600, height=400, background_color="white", colormap="viridis").generate(text_blob)
    axes[1, 1].imshow(wc, interpolation="bilinear")
    axes[1, 1].axis("off")
    axes[1, 1].set_title("Word Cloud of Comments")

    weekly = df.set_index("timestamp").groupby([pd.Grouper(freq="W"), "sentiment"]).size().unstack(fill_value=0)
    weekly = weekly.reindex(columns=order, fill_value=0)
    for s in order:
        axes[2, 0].plot(weekly.index, weekly[s], marker="o", label=s, color=SENTIMENT_COLORS[s])
    axes[2, 0].set_title("Sentiment Trend Over Time (Weekly)")
    axes[2, 0].set_ylabel("Number of Comments")
    axes[2, 0].tick_params(axis="x", rotation=30)
    axes[2, 0].legend(title="Sentiment")

    platform_score = df.groupby("platform")["sentiment_score"].mean().sort_values()
    colors = ["#e74c3c" if v < 0 else "#2ecc71" for v in platform_score.values]
    axes[2, 1].barh(platform_score.index, platform_score.values, color=colors)
    axes[2, 1].set_title("Average Sentiment Score by Platform")
    axes[2, 1].set_xlabel("Mean VADER Compound Score")
    axes[2, 1].axvline(0, color="black", linewidth=0.8)

    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight")
    display(fig)      # <-- forces the figure to render in the notebook output
    plt.close(fig)

    return out_path

In [23]:

def main():
    print("Task 1: Collecting comments...")
    raw_df = collect_comments(n=400)
    raw_df.to_csv("raw_comments.csv", index=False)
    print(f"  Collected {len(raw_df)} comments\n")

    print("Task 2: Cleaning and preprocessing...")
    clean_df = preprocess(raw_df)
    clean_df.to_csv("cleaned_comments.csv", index=False)
    print(f"  {len(clean_df)} comments remain after cleaning\n")

    print("Task 3: Running sentiment analysis...")
    sent_df = add_sentiment(clean_df)
    sent_df.to_csv("sentiment_comments.csv", index=False)
    print(sent_df["sentiment"].value_counts().to_string())
    print()

    print("Task 4: Identifying trending topics...")
    keyword_df, vectorizer, tfidf_matrix = find_top_keywords(sent_df)
    print("Top keywords:")
    print(keyword_df.head(10).to_string(index=False))
    topics = find_topics(tfidf_matrix, vectorizer)
    print("\nTopic clusters:")
    for topic, words in topics.items():
        print(f"  {topic}: {', '.join(words)}")
    print()

    print("Task 5: Building visualization dashboard...")
    out_path = build_dashboard(sent_df, keyword_df)
    print(f"  Dashboard saved to {out_path}")

    print("\nPipeline complete. Outputs: raw_comments.csv, cleaned_comments.csv, "
          "sentiment_comments.csv, opinion_dashboard.png")


if __name__ == "__main__":
    main()


Task 1: Collecting comments...
  Collected 400 comments

Task 2: Cleaning and preprocessing...
  400 comments remain after cleaning

Task 3: Running sentiment analysis...
sentiment
Positive    188
Neutral     118
Negative     94

Task 4: Identifying trending topics...
Top keywords:
     keyword     score
  smartwatch 24.281907
ultrablender 22.942774
   ecobottle 22.556784
     fitband 20.148649
         pro 19.499419
       clone 19.499419
     airpods 19.499419
     quality 15.778719
       works 13.074044
    purchase 13.027520

Topic clusters:
  Topic 1: fantastic, asked, service, customer, greatservice, brands, comparing, can
  Topic 2: regret, campaign, buy, don, convinced, saw, looks, perfectly
  Topic 3: purchase, ecobottle, warranty, come, happy, ultrablender, life, smartwatch
  Topic 4: recommend, highly, looking, anyone, five, shipping, stars, great

Task 5: Building visualization dashboard...
  Dashboard saved to opinion_dashboard.png

Pipeline complete. Outputs: raw_comment